In [ ]:
import config

## Смотрим на результаты работы BLAST на последовательностях из общего файла и находим список индексов последовательностей, которые нужно удалить, чтобы не было пересечений

In [ ]:
from collections import defaultdict

def clean_id(x):
    return x.split()[0]

# читаем BLAST результаты
pairs = []
with open(f"{config.DIR_BLAST}/results01_filtered.txt") as f:
    for line in f:
        parts = line.strip().split()
        q = clean_id(parts[0])
        s = clean_id(parts[1])
        pairs.append((q, s))

# строим граф
graph = defaultdict(set)
for a, b in pairs:
    graph[a].add(b)
    graph[b].add(a)

# поиск кластеров
visited = set()
clusters = []

def dfs(node, cluster):
    stack = [node]
    while stack:
        n = stack.pop()
        if n not in visited:
            visited.add(n)
            cluster.add(n)
            stack.extend(graph[n])

for node in graph:
    if node not in visited:
        cluster = set()
        dfs(node, cluster)
        clusters.append(cluster)

# выбираем кого удалить
remove = set()

for cluster in clusters:
    rep = sorted(cluster)[0]  # оставляем первый по алфавиту
    remove.update(cluster - {rep})

# сохраняем
with open(f"{config.DIR_REPBASE_PREPROCESSED}/remove_ids00.txt", "w") as f:
    for x in sorted(remove):
        f.write(x + "\n")

print(f"Удалить: {len(remove)} последовательностей")